# BoneNano step 3 — warm start and positions

Takes `code.npy` and `prb.npy` from **this folder's own stage 0** (`<root>/nfp`,
written by `step0.py`) and works out where CA2 sat during scan 86, then writes
the last three datasets `step6.py` needs.

This is the step that joins the two stages together, and it is where the
conventions matter most. Two in particular:

- **Stage 0's grid origin is arbitrary.** It anchored its aperture plane to
  scan 80's own random-walk centre, which says nothing about scan 86. So the
  positions here are measured *directly against a reference built from stage
  0's own output* — that recovers the walk and the constant offset together,
  in one shot, with no cumulative chain. (Stage 0 chains; this stage does not.
  The difference is deliberate.)
- **The sign is negated**, matching every phascade experiment on this engine
  family.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('../../src'))
import numpy as np, cupy as cp, h5py
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable

def cbshow(ax, img, title, cmap='gray', **kw):
    im = ax.imshow(np.asarray(img), cmap=cmap, **kw)
    ax.set_title(title, fontsize=9)
    cax = make_axes_locatable(ax).append_axes('right', size='5%', pad=0.05)
    ax.figure.colorbar(im, cax=cax)

print('GPU:', cp.cuda.runtime.getDeviceProperties(0)['name'].decode())

## Config

In [ ]:
# --- everything comes from config_step6.conf. Edit THAT file, not this cell.
from holotomocupy.config import parse_args_coded
CFG = parse_args_coded('config_step6.conf')

RAW_DIR = '/data3/cekmekci/bone_nano_holotomo'   # raw_dir; parse_args_coded has no key for it
OUT_H5  = CFG.in_file

DARK_H5  = f'{RAW_DIR}/scan-0067_orca.h5'   # dark
FLAT_H5  = f'{RAW_DIR}/scan-0068_orca.h5'   # flat: no CA1, no CA2
GRID_H5  = f'{RAW_DIR}/scan-0086_orca.h5'   # CA1 + CA2 + rotating sample, 95 GB
META_H5  = f'{RAW_DIR}/scan-0086.h5'        # companion: rotation angle
ORCA     = '/entry/instrument/orca/data'
THETA_DS = '/entry/instrument/tom_ry/value'   # DEGREES

ENERGY  = CFG.energy
Z1      = CFG.z1                 # focus -> SAMPLE
Z_CODE  = CFG.z1_code            # focus -> CA2
Z_DET   = CFG.focusdetectordistance  # focus -> detector  (TODO: confirm per beamtime)
DET_PIX = CFG.detector_pixelsize # UNBINNED
nz, n   = CFG.crop_nz, CFG.crop_n   # NATIVE crop resolution -- NOT CFG.nz/CFG.n,
                                    # which are step 6's own BINNED solve grid.
                                    # Steps 1-5 never bin anything; that is
                                    # CodedReader's job alone at step 6 (same
                                    # as AtomiumS2_coded). See the README's
                                    # "steps 1-5 never bin" section.
CROP    = (nz, n)
DEZINGER = (3, 0.8)
PREVIEW_BIN = CFG.bin   # cosmetic ONLY -- shrinks a notebook thumbnail in
                        # PDATA_CHECK below. Not used in any registration or
                        # writing math; see the note below.

# NOT read for any registration/writing math: CFG.ntheta/start_theta/stride
# (preprocessing never subsamples the angles -- see GEOM_THETA below) or
# CFG.bin/CFG.nz/CFG.n (preprocessing never bins -- see above). All four are
# the step6.py/CodedReader read-time parameters, not preprocessing's.

print(f'{ENERGY} keV   z1(sample) {Z1} m   z1_code(CA2) {Z_CODE} m   z_det {Z_DET} m')
print(f'crop {CROP}, native resolution -- step 6 bins to {CFG.nz} x {CFG.n} at bin={CFG.bin}')
assert Z_CODE < Z1, 'CA2 must be BEFORE the sample for RecCoded'

## Geometry and angle selection

In [ ]:
with h5py.File(GRID_H5, 'r') as f:
    NANG, NY, NX = f[ORCA].shape
cy, cx = NY // 2, NX // 2
SL = (slice(cy - CROP[0] // 2, cy + CROP[0] // 2),
      slice(cx - CROP[1] // 2, cx + CROP[1] // 2))

with h5py.File(META_H5, 'r') as f:
    theta_all = np.asarray(f[THETA_DS], 'float64')
assert len(theta_all) == NANG, f'{len(theta_all)} angles vs {NANG} frames'

# ALL frames, always -- no angle subsampling at preprocessing time. Quick-look
# runs subsample later, at step 6, via config_step6.conf's own
# ntheta/start_theta/stride (CodedReader indexes directly into this fully
# stored array). Same convention AtomiumS2_coded uses: write everything once,
# select a subset only when solving.
NTHETA = NANG
sel = np.arange(NANG)
theta_deg = theta_all[sel]

mag       = Z_DET / Z1
voxelsize = DET_PIX / mag   # NATIVE voxel size -- no bin factor, steps 1-5 never bin

print(f'orca {NY} x {NX}, {NANG} frames; crop rows {SL[0].start}:{SL[0].stop} '
      f'cols {SL[1].start}:{SL[1].stop}')
print(f'theta: ALL {NTHETA} angles converted, [{theta_deg.min():.3f}, {theta_deg.max():.3f}] deg, '
      f'step {np.diff(theta_deg).mean():.4f} deg')
print(f'voxel {voxelsize*1e9:.2f} nm   magnification {mag:.3f}')
print('quick-look subsampling, if wanted, happens at step 6 via '
      "config_step6.conf's own ntheta/start_theta/stride")

plt.figure(figsize=(5, 3))
plt.plot(theta_all, '.', ms=2)
plt.xlabel('frame'); plt.ylabel('tom_ry [deg]'); plt.grid(alpha=.3)
plt.tight_layout(); plt.show()

## The warm start from stage 0

Stage 0 ran on the **same square crop, also unbinned** — steps 1-5 never bin
anything (see the README's "steps 1-5 never bin" section), so the stage-0
arrays are reused as-is, only cropped in case `config_step0.conf`'s plane was
ever made larger than this stage's crop.

Reusing them across the two planes is exact rather than approximate: the
code->sample distance carries a `(z1/z1_code)**2` factor, which is the Fresnel
scaling theorem's compensation for reinterpreting the array on a different
pixel grid.

In [ ]:
from holotomocupy.propagation import Propagation

# The warm start from stage 0. code.npy / prb.npy were fitted on the CA2-plane
# voxelsize; reusing them here on the unified sample-plane grid is EXACT, not
# approximate. RecCoded rescales the code->sample distance by (z1/z1_code)**2,
# which is the Fresnel scaling theorem's compensation for reinterpreting the
# array on a different pixel grid. (The phascade catomo notebook says the same
# thing about code_before_distances.)
# CFG.code_file points at <root>/nfp/code.npy, written by this folder's step0.py
code_nfp = np.load(CFG.code_file).astype('complex64')
prb_nfp  = np.load(CFG.code_file.replace('code.npy', 'prb.npy')).astype('complex64')
print('code.npy', code_nfp.shape, '  prb.npy', prb_nfp.shape)

def crop_center(a, my, mx):
    ky, kx = a.shape
    sy, sx = (ky - my) // 2, (kx - mx) // 2
    return a[sy:sy + my, sx:sx + mx]

def pad_center(a, my, mx, fill):
    ky, kx = a.shape
    if ky >= my and kx >= mx:
        return crop_center(a, my, mx)
    out = np.full((my, mx), fill, a.dtype)
    py, px = (my - ky) // 2, (mx - kx) // 2
    out[py:py + ky, px:px + kx] = a
    return out

# Stage 0 ran UNBINNED on the same square crop, exactly this resolution --
# step 3 is native too (no bin2 anywhere: see the README's "steps 1-5 never
# bin" section), so NO binning here, only a crop in case config_step0.conf
# was ever changed to a different (larger) plane.
code_c = crop_center(code_nfp, n, n).astype('complex64')
prb_c  = crop_center(prb_nfp,  n, n).astype('complex64')
print(f'central {n}x{n} taken for the reference (native resolution, no bin)')

## Build the reference speckle

Push stage 0's code and probe through the same forward chain the data went
through, then apply the same two divisions. What comes out is what CA2 *should*
look like on the detector, which is what the frames get registered against.

In [ ]:
# Build the reference speckle the frames are registered against: stage 0's
# code and probe pushed through the SAME forward chain the data went through,
# then the same two divisions. NATIVE resolution throughout -- n, nz here are
# CFG.crop_nz/CFG.crop_n, not step 6's binned grid.
#
# RecCoded computes d1/d2 internally from z1/z1_code/fdd; replicate that here
# so the reference is built on the geometry the solver will actually use.
z2  = Z_DET - Z1
distance = Z1 * z2 / Z_DET
z2c = Z1 - Z_CODE
distance_code = (Z_CODE * z2c) / (Z_CODE + z2c)
norm_mag_code = (Z1 / Z_CODE) / (Z_CODE / Z1)
distance_code = distance_code * norm_mag_code ** 2 * (Z_CODE / Z1) ** 2
print(f'd1 (code->sample) {distance_code*1e3:+.4f} mm   '
      f'd2 (sample->det) {distance*1e3:+.4f} mm')

wavelength = 1.24e-9 / ENERGY
P1 = Propagation(n, nz, 1, 1, wavelength, voxelsize, np.array([distance_code]))
P2 = Propagation(n, nz, 1, 1, wavelength, voxelsize, np.array([distance]))

g_prb, g_code = cp.asarray(prb_c), cp.asarray(code_c)
I_prb  = cp.abs(P2.D(P1.D(g_prb[None], 0), 0)[0]) ** 2
I_code = cp.abs(P2.D(P1.D((g_prb * g_code)[None], 0), 0)[0]) ** 2
ref_rd = I_code / (I_prb + 1e-6)   # kept on GPU -- POSCODE correlates every
                                   # frame against this, avoid a host round trip

fig, ax = plt.subplots(1, 3, figsize=(16, 3.8))
cbshow(ax[0], np.angle(code_c), 'stage-0 code, central crop (phase)')
cbshow(ax[1], cp.asnumpy(I_code), '|D2 D1 (prb*code)|^2')
cbshow(ax[2], cp.asnumpy(ref_rd), 'ref_rd = I_code / I_prb')
plt.tight_layout(); plt.show()

In [ ]:
def phase_corr(a, b):
    A, B = cp.fft.fft2(cp.asarray(a)), cp.fft.fft2(cp.asarray(b))
    cc = cp.fft.fftshift(cp.fft.ifft2(A * cp.conj(B)).real)
    i = cp.unravel_index(cp.argmax(cc), cc.shape)
    return (np.array([int(i[0]) - cc.shape[0] // 2,
                      int(i[1]) - cc.shape[1] // 2], 'float32'),
            float(cc.max() / (cc.std() + 1e-8)))

## Diagnostic: did the probe move between stage 0 and this scan?

CA2's random walk averages out of the theta-mean, leaving CA1 + the probe +
the rotation-averaged sample. Compare that against the stage-0 probe pushed
through the same forward model.

This is not a formality — `BoneNano_nfp`'s own README documents exactly this
check turning up a **non-negligible** shift between an earlier pair of scans
(69 -> 80), and that scan was abandoned as a result. This cell is how we would
know if the same thing happened between scan 80 and scan 86.

In [ ]:
# Did the probe (and CA1, which lives inside it) move between stage 0 (scan
# 80) and this scan (86)? Ported from phascade's own bone_nano notebook, cell
# 14 -- a diagnostic there too, measured and inspected, not auto-corrected.
# This is NOT a formality: BoneNano_nfp's own README documents exactly this
# check turning up a NON-negligible shift between an earlier pair of scans
# (69 -> 80), which is why scan 69 was abandoned outright rather than reused.
# The same thing could have happened between 80 and 86 -- this is how we'd know.
CHUNK = 32
with h5py.File(OUT_H5, 'r') as f:
    static_sum = np.zeros((nz, n), 'float64')
    for cs in range(0, NTHETA, CHUNK):
        ce = min(cs + CHUNK, NTHETA)
        static_sum += np.asarray(f['/exchange/pdata'][cs:ce], 'float32').sum(0, dtype='float64')
static = cp.asarray((static_sum / NTHETA).astype('float32'))   # CA2's walk averages
                                                                 # out; CA1 + probe +
                                                                 # rotation-averaged
                                                                 # bone survive

def hp(img, s=0.02):
    f = cp.fft.fftfreq(img.shape[0]).astype('float32')
    fx, fy = cp.meshgrid(f, f)
    lp = cp.exp(-(fx ** 2 + fy ** 2) / (2 * s ** 2)).astype('float32')
    return (img / (cp.fft.ifft2(cp.fft.fft2(img) * lp).real + 1e-3)).astype('float32')

I_prb_hp, static_hp = hp(I_prb), hp(static)
prb_shift, _ = phase_corr(I_prb_hp, static_hp)
rolled = cp.roll(I_prb_hp, (-int(round(prb_shift[0])), -int(round(prb_shift[1]))), axis=(0, 1))
print(f'probe shift (stage0 -> scan86): row {prb_shift[0]:.2f}  col {prb_shift[1]:.2f} px')
print(f'||diff|| before = {float(cp.linalg.norm(I_prb_hp - static_hp)):.3e}   '
      f'after = {float(cp.linalg.norm(rolled - static_hp)):.3e}')
if abs(prb_shift[0]) >= 1 or abs(prb_shift[1]) >= 1:
    print('  -> NON-NEGLIGIBLE. The stage-0 probe/CA1 may not be where this scan has it -- '
          'see BoneNano_nfp/README.md ("CA1 moved between scan 69 and scan 80") for what that cost there.')
else:
    print('  -> negligible, consistent with reusing the stage-0 probe unmodified.')

fig, ax = plt.subplots(1, 4, figsize=(20, 5))
cbshow(ax[0], cp.asnumpy(I_prb_hp), 'stage-0 probe at detector, high-pass')
cbshow(ax[1], cp.asnumpy(static_hp), 'scan 86 mean frame, high-pass')
cbshow(ax[2], cp.asnumpy(I_prb_hp - static_hp), 'difference, no shift')
cbshow(ax[3], cp.asnumpy(rolled - static_hp), f'difference, after {-prb_shift} px')
plt.tight_layout(); plt.show()

## Positions

Isolate CA2 with the two divisions, then correlate every frame against the
reference. `pos_obj` is zero plus the rotation-axis offset — the sample does
not translate, it rotates.

Watch the **confidence** trace. A frame where the peak is not far above the
background is a frame whose position is a guess.

In [ ]:
# Isolate CA2 exactly as stage 0 does: divide out the no-CA illumination, then
# the theta-mean. The mean removes CA1, the probe residual AND the
# rotation-averaged bone, leaving only the moving aperture.
#
# NATIVE resolution throughout -- no bin2 (see the README's "steps 1-5 never
# bin" section). Streamed in TWO passes to avoid materialising the whole
# (NTHETA, nz, n) stack, which would be ~67 GB at the full 4001 angles and
# native 2048 x 2048: pass 1 accumulates the mean of pdata/pref; pass 2
# re-reads each frame, applies both divisions, and correlates -- AND computes
# the consecutive-frame step used by the chain diagnostic below, as a
# side-product of this same pass rather than a separate read of the 63 GB file.
CHUNK = 32
with h5py.File(OUT_H5, 'r') as f:
    pref = np.asarray(f['/exchange/pref'], 'float32')
    rd_sum = np.zeros((nz, n), 'float64')
    for cs in range(0, NTHETA, CHUNK):
        ce = min(cs + CHUNK, NTHETA)
        chunk = np.asarray(f['/exchange/pdata'][cs:ce], 'float32') / (pref[None] + 1e-6)
        rd_sum += chunk.sum(0, dtype='float64')
rd_mean = (rd_sum / NTHETA).astype('float32')
print(f'pass 1/2: theta-mean of pdata/pref over all {NTHETA} angles')

pref_gpu    = cp.asarray(pref)
rd_mean_gpu = cp.asarray(rd_mean) + 1e-6
res = []
stepk = np.empty((NTHETA - 1, 2), 'float32')
prev_rd = None
with h5py.File(OUT_H5, 'r') as f:
    pdata_dset = f['/exchange/pdata']
    for k in range(NTHETA):
        rd_k = cp.asarray(pdata_dset[k], 'float32') / (pref_gpu + 1e-6) / rd_mean_gpu
        res.append(phase_corr(ref_rd, rd_k))
        if prev_rd is not None:
            stepk[k - 1] = phase_corr(prev_rd, rd_k)[0]
        prev_rd = rd_k
        if k % 500 == 0:
            print(f'  pass 2/2: {k}/{NTHETA}', flush=True)

# NEGATED, and registered DIRECTLY against ref_rd rather than chained. Stage 0
# anchored its grid to scan 80's own walk centre, so its origin says nothing
# about where CA2 sat in scan 86; correlating against the stage-0 reference
# recovers the walk AND that constant offset in one shot, with no cumulative
# drift. (This is where the holotomo stage differs from stage 0, which chains.)
pos_code = -np.stack([r[0] for r in res]).astype('float32')
conf = np.array([r[1] for r in res], 'float32')

# rot_axis_offset is documented in config_step6.conf as UNBINNED px -- added
# directly, no scale factor, since pos_obj here is NATIVE too.
pos_obj = np.zeros((NTHETA, 2), 'float32') + np.asarray(CFG.rot_axis_offset, 'float32')

print(f'offset to stage 0 = {np.round(np.median(pos_code, 0), 2)} px')
print(f'walk span         = {np.round(np.ptp(pos_code, 0), 1)} px')
print(f'confidence  median {np.median(conf):.1f}  min {conf.min():.1f} '
      f'(frame {int(conf.argmin())})')

fig, ax = plt.subplots(1, 3, figsize=(16, 3.8))
ax[0].plot(pos_code[:, 1], pos_code[:, 0], 'o', ms=3)
ax[0].set_xlabel('col [px]'); ax[0].set_ylabel('row [px]'); ax[0].set_aspect('equal')
ax[0].set_title('pos_code'); ax[0].grid(alpha=.3)
ax[1].plot(pos_code[:, 0], label='row'); ax[1].plot(pos_code[:, 1], label='col')
ax[1].set_xlabel('projection'); ax[1].legend(); ax[1].grid(alpha=.3)
ax[2].plot(conf, '.-'); ax[2].set_xlabel('projection')
ax[2].set_title('registration confidence (peak/std)'); ax[2].grid(alpha=.3)
plt.tight_layout(); plt.show()

## Diagnostic: does an independent chain registration agree?

Consecutive-frame cross-correlation, cumulative from a mid-scan anchor,
exactly as stage 0 does it. It never touches stage 0's code, so agreement
with the absolute positions above is evidence the registration is actually
correct, not just self-consistent.

In [ ]:
# Independent chain registration, exactly as stage 0 does it (consecutive-
# frame cross-correlation, cumulative from a mid-scan anchor), compared
# against the absolute registration above. It never touches stage 0's code,
# so agreement is the evidence that the absolute registration really is
# correct -- not just self-consistent. Ported from phascade's own bone_nano
# notebook, cell 20. Only the WALK is comparable: the chain's own DC offset
# is arbitrary, so it is matched to the absolute positions by median.
ipos = NTHETA // 2
chain = np.zeros((NTHETA, 2), 'float32')
for k in range(ipos):
    chain[k] = np.sum(stepk[k:ipos], axis=0)
for k in range(ipos + 1, NTHETA):
    chain[k] = -np.sum(stepk[ipos:k], axis=0)
chain = chain + np.median(pos_code - chain, axis=0)
resid = pos_code - chain

print(f'|absolute - chain|: median {np.round(np.median(np.abs(resid), 0), 2)} px   '
      f'95th pct {np.round(np.percentile(np.abs(resid), 95, 0), 2)} px')
print('  healthy: ~1 px (see BoneNano_nfp/README.md\'s own diagnostics table). '
      'Much larger means the absolute registration above is not actually locking onto CA2.')

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(chain[:, 1], chain[:, 0], '.', ms=3, alpha=0.6, label='chain')
ax[0].plot(pos_code[:, 1], pos_code[:, 0], 'o', ms=3, mfc='none', label='absolute')
ax[0].set_xlabel('col [px]'); ax[0].set_ylabel('row [px]'); ax[0].grid(alpha=.3)
ax[0].set_aspect('equal'); ax[0].legend()
ax[1].plot(resid[:, 0], label='row'); ax[1].plot(resid[:, 1], label='col')
ax[1].set_xlabel('projection'); ax[1].set_ylabel('absolute - chain [px]'); ax[1].grid(alpha=.3); ax[1].legend()
plt.tight_layout(); plt.show()

## Diagnostic: is the configured canvas big enough?

`ncode` must clear `n + 2*max|pos_code| + 2*EX + 32`, or the per-projection
code window reads past the edge of the plane. Same for `nobj`.

In [ ]:
# EX is a guard margin, in NATIVE px (this stage runs unbinned throughout).
# 16 binned px was the original margin at bin=3; scaled up to native px here
# so the same PHYSICAL margin survives whatever bin config_step6.conf uses.
EX = 16 * 2 ** CFG.bin

# HARD constraint: CodedReader bins code.npy by 2**bin and then asserts the
# result is exactly (ncode, ncode). So ncode is NOT free -- it is fixed by
# stage 0's npsi and config_step6.conf's own bin. Getting this wrong is a
# ValueError at step6 startup. This is the ONE place step 3 needs to know
# about step 6's bin -- as a forward-looking cross-check, not to do any
# binning itself.
import numpy as _np
_code_on_disk = _np.load(CFG.code_file, mmap_mode='r').shape[0]
ncode_at_bin = _code_on_disk >> CFG.bin
print(f'stage-0 code.npy is {_code_on_disk} px, NATIVE -- step 6 will bin it '
      f'by 2**{CFG.bin} -> {ncode_at_bin}')
print(f'   config ncode = {CFG.ncode}' +
      ('   OK' if CFG.ncode == ncode_at_bin else
       f'   MISMATCH -- set ncode={ncode_at_bin} or re-run stage 0 with a different npsi'))

# Margin check, entirely in NATIVE px: does the stage-0 code plane (and the
# NATIVE equivalent of config_step6.conf's nobj) clear the measured walk?
nobj_native = CFG.nobj << CFG.bin
need_code = n + 2 * int(np.ceil(np.abs(pos_code).max())) + 2 * EX + 32
need_obj  = n + 2 * int(np.ceil(np.abs(pos_obj).max()))  + 2 * EX + 32
print()
print(f'walk needs code plane >= {need_code} native px   '
      f'(stage-0 npsi is {_code_on_disk})')
print(f'walk needs nobj       >= {need_obj} native px    '
      f'(config nobj={CFG.nobj} at bin={CFG.bin} is {nobj_native} native px)')
if _code_on_disk < need_code:
    print('  -> stage 0 npsi is TOO SMALL for the walk measured here. Raise '
          'npsi in config_step0.conf and re-run stage 0.')
if nobj_native < need_obj:
    print(f'  -> raise nobj in config_step6.conf to at least '
          f'{int(np.ceil(need_obj / 2 ** CFG.bin / 32) * 32)} (at bin={CFG.bin})')
if _code_on_disk >= need_code and nobj_native >= need_obj:
    print('  -> grids are large enough')

## Write to `codedHT.h5`

In [ ]:
# Written DIRECTLY, no scale factor: pos_code and pos_obj are already NATIVE
# (unbinned) px, computed that way throughout this notebook (see the README's
# "steps 1-5 never bin" section) -- exactly the units CodedReader expects on
# disk, and exactly what AtomiumS2_coded's own step 3 does.
with h5py.File(OUT_H5, 'a') as f:
    for k, v in (('shifts_code', pos_code),
                 ('shifts_cor',  pos_obj)):
        if f'/exchange/{k}' in f:
            del f[f'/exchange/{k}']
        f.create_dataset(f'/exchange/{k}', data=v.astype('float32'))
    # The probe, UNBINNED, as CodedReader bins it itself.
    if '/exchange/prb' in f:
        del f['/exchange/prb']
    f.create_dataset('/exchange/prb', data=prb_nfp.astype('complex64'))
print('wrote /exchange/{shifts_code, shifts_cor, prb} (native, unbinned px)')
print()
print('codedHT.h5 now has everything step6.py needs:')
with h5py.File(OUT_H5, 'r') as f:
    for k in sorted(f['/exchange']):
        print(f'   /exchange/{k:24s} {f["/exchange"][k].shape}')
print()
print('Next:  mpirun -np <ngpus> ./set_affinity_gpu.sh python step6.py config_step6.conf')